[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/projects/04_time_series/project.ipynb)

# Project 04 — Dự báo doanh số theo ngày (Time Series)

> Một chuỗi cửa hàng cần dự báo doanh số **28 ngày tới** của từng ngày để lên kế hoạch nhập hàng và xếp ca nhân viên. Dữ liệu: doanh số 3 năm (`sales_daily.csv`) cùng lịch khuyến mãi (`promo`) và ngày lễ (`is_holiday`) — cả hai **biết trước** khi dự báo. Cách làm hiện tại: lấy doanh số tuần vừa rồi lặp lại cho 4 tuần tới (seasonal naive).

**Bạn sẽ làm**:
1. EDA: xu hướng, mùa vụ tuần/năm, tác động khuyến mãi.
2. Khung **backtest rolling-origin** (8 lần × 28 ngày) và các baseline: naive, seasonal naive, trung bình 28 ngày.
3. Mô hình đặc trưng (lag ≥ 28 ngày, lịch, khuyến mãi) với gradient boosting — **không rò rỉ** tương lai.
4. Khoảng dự báo 80%: quantile regression vs conformal theo thời gian — kiểm tra độ phủ thật.
5. Dự báo 28 ngày tiếp theo kèm khoảng; báo cáo.

**Tiêu chí đạt**: seasonal naive tốt hơn naive · MAE backtest của mô hình thấp hơn seasonal naive ít nhất 10% · độ phủ khoảng 80% (conformal) nằm trong [0.7, 0.95].

**Chương liên quan**: 15 (chia theo thời gian), 17 (đặc trưng thời gian), 25 (chuỗi thời gian, baseline, backtest), 12 (boosting), 37 (khoảng dự đoán, conformal).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/projects/04_time_series

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.utils import repo_path

sales = pd.read_csv(repo_path("data", "sample", "sales_daily.csv"), parse_dates=["date"]).set_index("date")
HORIZON = 28
print(sales.shape, sales.index.min().date(), "→", sales.index.max().date())
sales.head()

## Bước 1 — EDA

Tính: doanh số trung bình theo thứ trong tuần `by_dow` (0 = thứ Hai), theo tháng `by_month`, và **mức tăng do khuyến mãi** `promo_lift` = TB ngày có khuyến mãi ÷ TB ngày không khuyến mãi − 1. Vẽ chuỗi gốc kèm trung bình trượt 28 ngày.

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
by_dow = None
by_month = None
promo_lift = None
### END CODE HERE ###
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), gridspec_kw={"width_ratios": [3, 1, 1]})
sales.sales.plot(ax=axes[0], lw=0.6, alpha=0.6, label="doanh số")
sales.sales.rolling(28).mean().plot(ax=axes[0], color="C3", label="TB trượt 28 ngày")
axes[0].legend()
by_dow.plot(kind="bar", ax=axes[1], title="theo thứ (0 = T2)", rot=0)
by_month.plot(kind="bar", ax=axes[2], title="theo tháng", rot=0)
plt.tight_layout()
plt.show()
print(f"khuyến mãi làm doanh số tăng {promo_lift:.0%}")

In [ ]:
check_equal(len(by_dow), 7, "7 thứ trong tuần")
check_greater(by_dow.max() / by_dow.min(), 1.1, "có mùa vụ tuần rõ")
check_greater(promo_lift, 0.1, "khuyến mãi tăng doanh số > 10%")

## Bước 2 — Backtest và baseline

Một lần chia train/test không đủ cho chuỗi thời gian. **Rolling-origin**: chọn 8 điểm cắt cách nhau 28 ngày ở cuối chuỗi; với mỗi điểm cắt, chỉ dùng dữ liệu **trước** điểm cắt để dự báo 28 ngày sau đó.

Viết `backtest(forecast_fn, n_origins=8)`: `forecast_fn(history: DataFrame, future: DataFrame) -> np.ndarray` (độ dài 28; `future` chứa `promo`, `is_holiday` của 28 ngày tới, **không** có `sales`); trả về DataFrame một dòng/điểm cắt với cột `origin`, `mae`, và lưu dự báo vào cột `forecast`, giá trị thật `actual`. Sau đó cài 3 baseline: `naive` (giá trị cuối), `seasonal_naive` (7 ngày cuối lặp lại 4 lần), `mean28` (TB 28 ngày cuối).

In [ ]:
def backtest(forecast_fn, n_origins=8):
    ### START CODE HERE ### (≈ 9 dòng)
    return pd.DataFrame()
    ### END CODE HERE ###


def naive(history, future):
    ### START CODE HERE ### (≈ 1 dòng)
    return None
    ### END CODE HERE ###


def seasonal_naive(history, future):
    ### START CODE HERE ### (≈ 1 dòng)
    return None
    ### END CODE HERE ###


def mean28(history, future):
    ### START CODE HERE ### (≈ 1 dòng)
    return None
    ### END CODE HERE ###


results = {name: backtest(fn) for name, fn in [("naive", naive), ("seasonal naive", seasonal_naive), ("TB 28 ngày", mean28)]}
print(pd.DataFrame({name: r.mae.values for name, r in results.items()}, index=results["naive"].origin.dt.date).round(1))
print(pd.Series({name: r.mae.mean() for name, r in results.items()}).round(2).to_string())

In [ ]:
check_equal(len(results["naive"]), 8, "8 điểm cắt")
check_equal(len(results["naive"].forecast.iloc[0]), 28, "dự báo 28 ngày")
check_equal(str(results["naive"].origin.iloc[-1].date()), "2024-12-03", "điểm cắt cuối cùng")
check_less(results["seasonal naive"].mae.mean(), results["naive"].mae.mean(), "seasonal naive tốt hơn naive")

## Bước 3 — Mô hình đặc trưng

Dự báo **trực tiếp** 28 ngày: tại điểm cắt ta chỉ biết doanh số đến hôm nay, nên với ngày cần dự báo $t$ (xa nhất là hôm nay + 28) chỉ dùng được **lag ≥ 28**. Viết `make_features(frame)` cho DataFrame có cột `sales` (có thể NaN ở tương lai), `promo`, `is_holiday`: `dow`, `month`, `dayofyear`, `promo`, `is_holiday`, `t` (chỉ số thời gian), `lag28`, `lag35`, `lag42`, `lag364`, `roll28_lag28` (TB 28 ngày kết thúc 28 ngày trước). Rồi `boosting_forecast(history, future)`: ghép history + future, tạo đặc trưng, huấn luyện `HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=0)` trên các dòng history có đủ đặc trưng, dự báo cho future.

In [ ]:
def make_features(frame: pd.DataFrame) -> pd.DataFrame:
    ### START CODE HERE ### (≈ 10 dòng)
    return pd.DataFrame(index=frame.index)
    ### END CODE HERE ###


def boosting_forecast(history, future):
    ### START CODE HERE ### (≈ 6 dòng)
    return None
    ### END CODE HERE ###


results["boosting"] = backtest(boosting_forecast)
summary = pd.Series({name: r.mae.mean() for name, r in results.items()})
print(summary.round(2).to_string())
print(f"cải thiện so với seasonal naive: {1 - summary['boosting'] / summary['seasonal naive']:.0%}")

In [ ]:
probe = sales.iloc[:400].copy()
base_features = make_features(probe)
probe.loc[probe.index[-27:], "sales"] = -999                           # làm hỏng 27 ngày cuối
check_true(base_features.iloc[-1].equals(make_features(probe).iloc[-1]), "đặc trưng của một ngày không dùng doanh số 27 ngày gần nhất (không rò rỉ)")
check_less(summary["boosting"], 0.9 * summary["seasonal naive"], "tốt hơn seasonal naive ít nhất 10%")

## Bước 4 — Khoảng dự báo 80%

Nhập hàng cần biết **khoảng**, không chỉ một con số. Hai cách:
- **Quantile regression**: huấn luyện thêm hai mô hình với `loss="quantile"`, `quantile=0.1` và `0.9`.
- **Conformal theo thời gian**: dùng phần dư của **các lần backtest trước** — khoảng = dự báo ± phân vị 80% của |phần dư| — (chương 37).

Tính độ phủ thật trên backtest: `coverage_quantile` (trung bình 8 điểm cắt), và `coverage_conformal` — với 14 điểm cắt, chỉ đánh giá các điểm cắt có ≥ 4 lần backtest trước đó (dùng phần dư của chúng).

In [ ]:
def quantile_forecast(history, future, q):
    frame = pd.concat([history, future.assign(sales=np.nan)])
    features = make_features(frame)
    train = features.iloc[:len(history)].dropna()
    model = HistGradientBoostingRegressor(loss="quantile", quantile=q, max_iter=300, learning_rate=0.05, random_state=0)
    return model.fit(train, history.sales.loc[train.index]).predict(features.iloc[len(history):])


### START CODE HERE ### (≈ 10 dòng)
coverage_quantile, coverage_conformal = None, None
### END CODE HERE ###
print(f"độ phủ thật của khoảng 80% — quantile regression: {coverage_quantile:.0%} | conformal theo thời gian: {coverage_conformal:.0%}")

In [ ]:
check_less(coverage_quantile, 0.75, "quantile regression phủ thiếu (quá tự tin)")
check_true(0.7 <= coverage_conformal <= 0.95, "conformal phủ gần mức danh nghĩa")

## Bước 5 — Dự báo 28 ngày tới

Giả sử 28 ngày sau 31/12/2024 không có khuyến mãi; ngày 1/1/2025 là ngày lễ. Tạo `future` (index ngày, cột `promo`, `is_holiday`), dự báo `final_forecast` bằng `boosting_forecast(sales, future)`, khoảng 80% bằng phân vị |phần dư| của **toàn bộ** backtest dài → `q80_all`. Vẽ 90 ngày cuối + dự báo + khoảng.

In [ ]:
### START CODE HERE ### (≈ 5 dòng)
future_index = None
future = None
final_forecast = None
q80_all = None
### END CODE HERE ###
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(sales.index[-90:], sales.sales.iloc[-90:], label="thực tế")
ax.plot(future_index, final_forecast, color="C3", label="dự báo")
ax.fill_between(future_index, final_forecast - q80_all, final_forecast + q80_all, color="C3", alpha=0.2, label="khoảng 80%")
ax.legend()
plt.show()
print(f"tổng doanh số dự báo 28 ngày: {final_forecast.sum():,.0f} | khoảng 80% mỗi ngày: ± {q80_all:.0f}")

In [ ]:
check_equal(len(final_forecast), 28, "28 ngày")
check_true(np.isfinite(final_forecast).all(), "không có NaN")
check_true(future.is_holiday.sum() == 1, "đánh dấu ngày lễ")

## Bước 6 — Báo cáo

Viết 5–8 câu cho bộ phận kế hoạch: phương pháp, độ chính xác backtest so với cách làm hiện tại, cách đọc khoảng dự báo, giả định (lịch khuyến mãi), hạn chế, đề xuất (dự báo lại hằng tuần, theo dõi sai số).

✍️ Báo cáo: